In [ ]:
# Run once. Uses only PyPI packages — no external APIs.
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--break-system-packages", "-q", *pkgs])

pip_install([
    "torch",
    "pandas",
    "numpy",
    "nltk",
    "bert-score",
    "sentencepiece",
    "matplotlib",
    "tqdm",
])

import nltk
nltk.download("punkt", quiet=True)
print("All dependencies installed.")


In [ ]:
import os, random, time, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# >>> EDIT THIS to point at the folder where you downloaded the assignment dataset <<<
DATA_DIR = "./data"

VOCAB_SIZE_SA = 8000      # SentencePiece vocab size, Sanskrit side
VOCAB_SIZE_EN = 6000      # SentencePiece vocab size, English side
MAX_LEN       = 60        # max tokens per sentence (after sub-word tokenization)
EMB_DIM       = 256
HID_DIM       = 512
ENC_LAYERS    = 2
DEC_LAYERS    = 1
DROPOUT       = 0.3
BATCH_SIZE    = 64
EPOCHS        = 25
LR            = 3e-4
CLIP          = 1.0
LABEL_SMOOTH  = 0.1
BEAM_WIDTH    = 5
PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"


In [ ]:
def load_split(split):
    sa = pd.read_csv(os.path.join(DATA_DIR, f"{split}_sa.csv"))
    en = pd.read_csv(os.path.join(DATA_DIR, f"{split}_en.csv"))
    sa.columns = [c.strip() for c in sa.columns]
    en.columns = [c.strip() for c in en.columns]
    df = pd.merge(sa, en, on="Source_id", how="inner")
    df = df.rename(columns={"Sentence_sa": "sa", "Sentence_en": "en"})
    df = df.dropna(subset=["sa", "en"]).reset_index(drop=True)
    return df

if os.path.isdir(DATA_DIR):
    train_df = load_split("train")
    dev_df   = load_split("dev")
    test_df  = load_split("test")
    print(f"train={len(train_df)}  dev={len(dev_df)}  test={len(test_df)}")
    train_df.head()
else:
    print(f"[WARNING] DATA_DIR='{DATA_DIR}' not found. "
          "Download the dataset from the assignment link and update DATA_DIR before running the rest of the notebook.")
    train_df = pd.DataFrame(columns=["Source_id", "sa", "en"])
    dev_df   = pd.DataFrame(columns=["Source_id", "sa", "en"])
    test_df  = pd.DataFrame(columns=["Source_id", "sa", "en"])


In [ ]:
import sentencepiece as spm

os.makedirs("spm", exist_ok=True)

def write_corpus(series, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in series.astype(str):
            f.write(s.strip() + "\n")

def train_spm(text_path, model_prefix, vocab_size):
    spm.SentencePieceTrainer.train(
        input=text_path,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=1.0,          # important for Devanagari
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece=PAD, unk_piece=UNK, bos_piece=SOS, eos_piece=EOS,
    )

if len(train_df) > 0:
    write_corpus(train_df["sa"], "spm/train_sa.txt")
    write_corpus(train_df["en"], "spm/train_en.txt")
    train_spm("spm/train_sa.txt", "spm/sa", VOCAB_SIZE_SA)
    train_spm("spm/train_en.txt", "spm/en", VOCAB_SIZE_EN)

    sp_sa = spm.SentencePieceProcessor(model_file="spm/sa.model")
    sp_en = spm.SentencePieceProcessor(model_file="spm/en.model")
    print("SA vocab:", sp_sa.vocab_size(), " EN vocab:", sp_en.vocab_size())


In [ ]:
def encode(sp, text, max_len=MAX_LEN):
    ids = [SOS_ID] + sp.encode(str(text), out_type=int)[: max_len - 2] + [EOS_ID]
    return ids

class TranslationDataset(Dataset):
    def __init__(self, df, sp_sa, sp_en):
        self.src = [encode(sp_sa, s) for s in df["sa"]]
        self.tgt = [encode(sp_en, s) for s in df["en"]]
        self.ids = df["Source_id"].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx], self.ids[idx]

def collate_fn(batch):
    srcs, tgts, ids = zip(*batch)
    src_lens = [len(s) for s in srcs]
    tgt_lens = [len(t) for t in tgts]
    max_s, max_t = max(src_lens), max(tgt_lens)
    src_pad = torch.full((len(batch), max_s), PAD_ID, dtype=torch.long)
    tgt_pad = torch.full((len(batch), max_t), PAD_ID, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, : len(s)] = torch.tensor(s)
        tgt_pad[i, : len(t)] = torch.tensor(t)
    return src_pad, torch.tensor(src_lens), tgt_pad, torch.tensor(tgt_lens), list(ids)

if len(train_df) > 0:
    PAD_ID, UNK_ID, SOS_ID, EOS_ID = sp_sa.pad_id(), sp_sa.unk_id(), sp_sa.bos_id(), sp_sa.eos_id()

    train_ds = TranslationDataset(train_df, sp_sa, sp_en)
    dev_ds   = TranslationDataset(dev_df,   sp_sa, sp_en)
    test_ds  = TranslationDataset(test_df,  sp_sa, sp_en)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
    dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    print("Batches -> train:", len(train_loader), "dev:", len(dev_loader), "test:", len(test_loader))


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if n_layers > 1 else 0.0)
        self.fc_h = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lens):
        emb = self.dropout(self.embedding(src))
        packed = nn.utils.rnn.pack_padded_sequence(emb, src_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, c) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        def merge_directions(state):
            # state: (n_layers*2, B, hid) -> take last layer's fwd/bwd, project to (B, hid)
            fwd, bwd = state[-2], state[-1]
            return torch.tanh(self.fc_h(torch.cat([fwd, bwd], dim=-1)))

        h_top = torch.tanh(self.fc_h(torch.cat([h[-2], h[-1]], dim=-1))).unsqueeze(0)
        c_top = torch.tanh(self.fc_c(torch.cat([c[-2], c[-1]], dim=-1))).unsqueeze(0)
        return outputs, (h_top, c_top)


class BahdanauAttention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.W_enc = nn.Linear(hid_dim * 2, hid_dim, bias=False)
        self.W_dec = nn.Linear(hid_dim, hid_dim, bias=False)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs, src_mask):
        # dec_hidden: (B, hid)  enc_outputs: (B, S, 2*hid)
        score = self.v(torch.tanh(self.W_dec(dec_hidden).unsqueeze(1) + self.W_enc(enc_outputs))).squeeze(-1)
        score = score.masked_fill(src_mask == 0, float("-inf"))
        attn_weights = F.softmax(score, dim=-1)              # (B, S)
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)  # (B, 2*hid)
        return context, attn_weights


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.attention = BahdanauAttention(hid_dim)
        self.rnn = nn.LSTM(emb_dim + hid_dim * 2, hid_dim, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hid_dim * 3 + emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_tok, hidden, cell, enc_outputs, src_mask):
        emb = self.dropout(self.embedding(input_tok)).unsqueeze(1)        # (B,1,E)
        context, attn_w = self.attention(hidden[-1], enc_outputs, src_mask)
        rnn_in = torch.cat([emb, context.unsqueeze(1)], dim=-1)
        out, (hidden, cell) = self.rnn(rnn_in, (hidden, cell))
        out = out.squeeze(1)
        logits = self.fc_out(torch.cat([out, context, emb.squeeze(1)], dim=-1))
        return logits, hidden, cell, attn_w


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder, self.decoder, self.device = encoder, decoder, device

    def make_src_mask(self, src):
        return (src != PAD_ID)

    def forward(self, src, src_lens, tgt, teacher_forcing_ratio=0.5):
        B, T = tgt.shape
        vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(B, T, vocab_size, device=self.device)

        enc_outputs, (h, c) = self.encoder(src, src_lens)
        src_mask = self.make_src_mask(src)
        input_tok = tgt[:, 0]  # <sos>

        for t in range(1, T):
            logits, h, c, _ = self.decoder(input_tok, h, c, enc_outputs, src_mask)
            outputs[:, t] = logits
            teacher_force = random.random() < teacher_forcing_ratio
            input_tok = tgt[:, t] if teacher_force else logits.argmax(-1)
        return outputs

if len(train_df) > 0:
    encoder = Encoder(sp_sa.vocab_size(), EMB_DIM, HID_DIM, ENC_LAYERS, DROPOUT)
    decoder = Decoder(sp_en.vocab_size(), EMB_DIM, HID_DIM, DEC_LAYERS, DROPOUT)
    model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {n_params:,}")


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0.0
    for src, src_lens, tgt, tgt_lens, _ in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        src_lens = src_lens.to(DEVICE)
        optimizer.zero_grad()
        output = model(src, src_lens, tgt, teacher_forcing_ratio=0.5)
        output_dim = output.shape[-1]
        loss = criterion(output[:, 1:].reshape(-1, output_dim), tgt[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

@torch.no_grad()
def evaluate_loss(model, loader, criterion):
    model.eval()
    epoch_loss = 0.0
    for src, src_lens, tgt, tgt_lens, _ in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        src_lens = src_lens.to(DEVICE)
        output = model(src, src_lens, tgt, teacher_forcing_ratio=0.0)
        output_dim = output.shape[-1]
        loss = criterion(output[:, 1:].reshape(-1, output_dim), tgt[:, 1:].reshape(-1))
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

if len(train_df) > 0:
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTH)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    train_losses, dev_losses = [], []
    best_dev_loss = float("inf")
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, CLIP)
        dv_loss = evaluate_loss(model, dev_loader, criterion)
        scheduler.step(dv_loss)
        train_losses.append(tr_loss); dev_losses.append(dv_loss)
        if dv_loss < best_dev_loss:
            best_dev_loss = dv_loss
            torch.save(model.state_dict(), "best_model.pt")
        print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | dev_loss={dv_loss:.4f} | "
              f"time={time.time()-t0:.1f}s")


In [ ]:
import matplotlib.pyplot as plt

if len(train_df) > 0:
    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label="train loss")
    plt.plot(dev_losses, label="dev loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Training curves")
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
@torch.no_grad()
def beam_search_decode(model, src, src_len, sp_en, beam_width=BEAM_WIDTH, max_len=MAX_LEN, len_penalty=0.7):
    model.eval()
    src = src.unsqueeze(0).to(DEVICE)
    src_len_t = torch.tensor([src_len]).to(DEVICE)
    enc_outputs, (h, c) = model.encoder(src, src_len_t)
    src_mask = model.make_src_mask(src)

    beams = [([SOS_ID], 0.0, h, c)]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for seq, score, h_b, c_b in beams:
            if seq[-1] == EOS_ID:
                completed.append((seq, score))
                continue
            input_tok = torch.tensor([seq[-1]], device=DEVICE)
            logits, h_n, c_n, _ = model.decoder(input_tok, h_b, c_b, enc_outputs, src_mask)
            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)
            topk_lp, topk_idx = log_probs.topk(beam_width)
            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                new_beams.append((seq + [idx], score + lp, h_n, c_n))
        if not new_beams:
            break
        new_beams.sort(key=lambda x: x[1] / (len(x[0]) ** len_penalty), reverse=True)
        beams = new_beams[:beam_width]
        if all(b[0][-1] == EOS_ID for b in beams):
            completed.extend([(b[0], b[1]) for b in beams])
            break

    completed.extend([(b[0], b[1]) for b in beams])
    completed.sort(key=lambda x: x[1] / (len(x[0]) ** len_penalty), reverse=True)
    best_seq = completed[0][0]
    ids = [t for t in best_seq if t not in (SOS_ID, EOS_ID, PAD_ID)]
    return sp_en.decode(ids)

def greedy_decode(model, src, src_len, sp_en, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src_b = src.unsqueeze(0).to(DEVICE)
        src_len_t = torch.tensor([src_len]).to(DEVICE)
        enc_outputs, (h, c) = model.encoder(src_b, src_len_t)
        src_mask = model.make_src_mask(src_b)
        input_tok = torch.tensor([SOS_ID], device=DEVICE)
        out_ids = []
        for _ in range(max_len):
            logits, h, c, _ = model.decoder(input_tok, h, c, enc_outputs, src_mask)
            next_tok = logits.argmax(-1)
            if next_tok.item() == EOS_ID:
                break
            out_ids.append(next_tok.item())
            input_tok = next_tok
        return sp_en.decode(out_ids)


In [ ]:
def translate_dataset(model, ds, sp_en, beam_width=BEAM_WIDTH):
    model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
    model.eval()
    preds, ids = [], []
    for src, tgt, sid in zip(ds.src, ds.tgt, ds.ids):
        pred = beam_search_decode(model, torch.tensor(src), len(src), sp_en, beam_width=beam_width)
        preds.append(pred)
        ids.append(sid)
    return ids, preds

if len(train_df) > 0:
    t0 = time.time()
    test_ids, test_preds = translate_dataset(model, test_ds, sp_en, beam_width=BEAM_WIDTH)
    inference_time = time.time() - t0
    print(f"Inference time on {len(test_ids)} test sentences: {inference_time:.2f}s "
          f"({inference_time/len(test_ids)*1000:.1f} ms/sentence)")


In [ ]:
from nltk.translate.bleu_score import corpus_bleu
from bert_score import score as bert_score_fn

if len(train_df) > 0:
    references = [[r.split()] for r in test_df["en"].tolist()]
    hypotheses = [p.split() for p in test_preds]

    bleu = corpus_bleu(references, hypotheses)  # default NLTK BLEU, no custom weights
    print(f"Corpus BLEU: {bleu:.4f}")

    P, R, F1 = bert_score_fn(test_preds, test_df["en"].tolist(), lang="en", rescale_with_baseline=True)
    bertscore_f1 = F1.mean().item()
    print(f"BERTScore (F1, rescaled): {bertscore_f1:.4f}")


In [ ]:
if len(train_df) > 0:
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters : {n_params:,}")
    print(f"Total inference time (test set, beam_width={BEAM_WIDTH}): {inference_time:.2f} s")
    print(f"Avg. inference time / sentence: {inference_time/len(test_ids)*1000:.2f} ms")

    metrics_summary = {
        "bleu": bleu,
        "bertscore_f1": bertscore_f1,
        "inference_time_sec": inference_time,
        "num_parameters": n_params,
    }
    with open("metrics_summary.json", "w") as f:
        json.dump(metrics_summary, f, indent=2)
    print(json.dumps(metrics_summary, indent=2))


In [ ]:
if len(train_df) > 0:
    submission = pd.DataFrame({"Source_id": test_ids, "Sentence_en": test_preds})
    submission.to_csv("submission.csv", index=False, encoding="utf-8")
    print("Saved submission.csv with", len(submission), "rows")
    submission.head()


In [ ]:
if len(train_df) > 0:
    sample_idx = random.sample(range(len(test_df)), min(8, len(test_df)))
    for i in sample_idx:
        src_text = test_df.iloc[i]["sa"]
        ref_text = test_df.iloc[i]["en"]
        pred_text = test_preds[i]
        print("SRC :", src_text)
        print("REF :", ref_text)
        print("PRED:", pred_text)
        print("-" * 60)

    # NOTE: After running on the real dataset, replace the print loop above with a markdown
    # table in the report (Section "Translation Examples") and manually annotate each example
    # with an error category, e.g.:
    #  1. Omission        - a content word from the source is dropped in the prediction
    #  2. Mistranslation   - wrong lexical choice for a Sanskrit root/compound
    #  3. Word-order error - correct words, non-fluent English ordering
    #  4. Agreement error  - subject-verb / number agreement mistakes
    #  5. Over-generation  - repeated or hallucinated tokens (common in early-stopped beams)
